# Data cleaning for the Index of Multiple Deprivation dataset

1 hour 10 mins morning 11th June

start 1705              11 th june evening 

## Data sources

The Index of Multiple Deprivation (IMD) dataset was published by the Ministry of Housing, Communities & Local Government (MHCLG) and was constructed by Oxford Consultants for Social Inclusion (OCSI) and Deprivation.org. This dataset was downloaded from the GOV.UK website, as an Excel file (.xlsx), using the link 'https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025' on 9th June 2026. The official statistics on the source website were published on 30 October 2025 and last updated on 17 November 2025.


An additional dataset has also been utilised, to support the preprocessing of the IMD dataset, the necessity of which is detailed fully later in this notebook. In brief, the dataset is required for standardising the location data of the IMD dataset, to allow comparison to other datasets (such as the ‘Police recorded crime open data tables, year ending March 2013 onwards’ dataset) and improve overall ease of understanding.

The 'Local Authority District (December 2024) to LAU1 to ITL3 to ITL2 to ITL1 (January 2025) Lookup in the UK' dataset was published by the Office for National Statistics (ONS). The dataset was downloaded from the National Data Library website, as a csv file (.csv), using the link 'https://www.data.gov.uk/dataset/2b47adcc-62b6-4dd3-a6cb-271b3035e9fd/local-authority-district-december-2024-to-lau1-to-itl3-to-itl2-to-itl1-january-2025-lookup-in-t' on 9th June 2026. The dataset was last updated on 09 December 2025 and the csv file was added to the website on 12 January 2025. 


 - merge datasets on the LADs column (so that the columns are then LSOA, LAD, IOD IOD, ITL1, ITL2)
 - dataframes and panda

## Requirements:
 - pandas
 - numpy
 - openpyxl

In [1]:
import pandas as pd
import numpy as np
import openpyxl as op

## Pre-cleaning data check

In [ ]:
# Uploading the IMD dataset
df = pd.read_excel('2025_index_of_multiple_deprivation.xlsx',
                  sheet_name='IMD25')

# Visually checking the first and last 5 rows of data in the IMD dataset
# Including key information, such as total number of rows, column names and a sample of data
print(f'First five rows of data:\n{df.head(5)}')
print(f'\nLast five rows of data:\n{df.tail(5)}')

# Shape of dataset - to confirm the number of rows and columns
print(f'\nPre-cleaning IMD dataset shape: {df.shape}')

# Summary statistics pre-cleaning - notice that this only provides stats for numerical data (the IMD rank and IMD decile)
print(f'\nPre-cleaning IMD summary statistics:\n{df.describe()}')


## Data Cleaning

Data quality checks, including:
 - Missing values
 - Data types
 - Unrealistic/unexpected values
 - Duplicate values

In [ ]:
# Copying the IMD dataset, in order to preserve the original dataset
df_copy = df.copy()

# Checking for missing values, with none found across all columns
missing_values = df_copy.isnull().sum()
print(f'Total missing values for the Index of Multiple Deprivation \
(IMD) dataset: {missing_values.sum()}')

# Investigating data types and assessing if they are the most appropriate for the values
print(f'\nAll data types for the IMD dataset: \nLSOA Code: {df_copy['LSOA code (2021)'].dtypes}\
\nLSOA Name: {df_copy['LSOA name (2021)'].dtypes}\nLAD Code: {df_copy['Local Authority District code (2024)'].dtypes}\
\nLAD Name: {df_copy['Local Authority District name (2024)'].dtypes}\nIMD Rank: \
{df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'].dtypes}\
\nIMD Decile: {df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA'].dtypes}')

# Checking that the IMD rank is between 1 and 33755, and that the IMD decile is between 1 and 10
if df_copy[(df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'] < 1) | \
    (df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']> 33755)].shape[0]:
    print(f'\nUnexpected IMD Rank values:\n{df_copy[(df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']\
    < 1) | (df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']> 33755)].values}')
else:
    print('\nNo unexpected IMD Rank values.')

if df_copy[(df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA'] < 1) | \
    (df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA']> 10)].shape[0]:
    print(f'\nUnexpected IMD Decile values:\n{df_copy[(df_copy['Index of Multiple Deprivation (IMD) \
    Decile (where 1 is most deprived 10% of LSOA'] < 1) | (df_copy['Index of Multiple Deprivation (IMD) Decile \
    (where 1 is most deprived 10% of LSOA']> 10)].values}')
else:
    print('\nNo unexpected IMD Decile values.')

All data types are suitable:
 - The LSOA code, LSOA name, LAD code and LAD name columns are all strings, which is appropriate as they contain either solely letters or both numbers and letters
 - The IMD rank and IMD decile are integers, which is the most appropriate data type for these numerical values 

The LSOA code, LSOA name and LAD name columns have been removed from the IMD dataset. This decision was made because these columns are superfluous to our needs, as the LAD code will be used to standardise location data to ITLs, and the LSOA codes, LSOA names and LAD names are, therefore, not required. 

In [13]:
# Remove unnecessary columns, such as the LSOA codes because unnecessary for analysis
df_imd = df.drop(['LSOA code (2021)', 'LSOA name (2021)', ''], axis=1)

# Shape of dataset - to confirm the columns have been deleted successfully
print(f'IMD dataset shape: {df_imd.shape}')

# Check for outliers in the dataset
# check for outliers with IQR method and remove outliers?

# Check for duplicates (see slides)


IMD dataset shape: (33755, 4)


## Standardising locations 

Mapping Local Authority District (LAD) codes against International Territorial Levels (ITLs), specifically ITL1 and ITL2.

I decided to use the LAD codes to map to the ITLs, rather than the LSOAs, as this approach is facilitated by the Office for National Statistics (ONS), which have published a 'Local Authority District (December 2024) to LAU1 to ITL3 to ITL2 to ITL1 (January 2025) Lookup in the UK' dataset. The LAD codes and names in the IMD dataset are for 2024, whilst the ITLs we decided to use are for 2025. Therefore, the ONS dataset chosen is the most appropriate for this task.

The ITL1 and ITL2 regions can then be utilised across the datasets for this project, ensuring that location data is directly comparable. This is necessary as, for example, the police dataset utilises police districts that are not directly comparable to LSOAs or LADs.

In [15]:
# Uploading the LAD to ITL lookup dataset
df_itl = pd.read_csv('LAD_2024_to_ITL_2025.csv')

# Chcking the LAD to ITL lookup dataset
print(f'\nPre-cleaning IMD dataset shape: {df_itl.shape}')

print(df_itl.head())


Pre-cleaning IMD dataset shape: (362, 11)
  ITL125CD              ITL125NM ITL225CD     ITL225NM ITL325CD  \
0      TLC  North East (England)     TLC3  Tees Valley    TLC31   
1      TLC  North East (England)     TLC3  Tees Valley    TLC31   
2      TLC  North East (England)     TLC3  Tees Valley    TLC32   
3      TLC  North East (England)     TLC3  Tees Valley    TLC32   
4      TLC  North East (England)     TLC3  Tees Valley    TLC33   

                          ITL325NM   LAU125CD              LAU125NM  \
0  Hartlepool and Stockton-on-Tees  E06000001            Hartlepool   
1  Hartlepool and Stockton-on-Tees  E06000004      Stockton-on-Tees   
2                   South Teesside  E06000002         Middlesbrough   
3                   South Teesside  E06000003  Redcar and Cleveland   
4                       Darlington  E06000005            Darlington   

     LAD24CD               LAD24NM  ObjectId  
0  E06000001            Hartlepool         1  
1  E06000004      Stockton-on-Tee

The LAD_2024_to_ITL_2025 ONS dataset contains location data for England, Wales, Scotland and Northern Ireland, whereas the Index for Multiple Deprivation dataset only contains data for England. Therefore, I have decided to use a ** merge, to retain all location data from the Index for Multiple Deprivation dataset and not pull across any ITLs that aren't required (for example, Wales, Scotland and Northern Ireland).

In [ ]:
# Merging the IMD dataset and the LAD to ITL lookup dataset
df_clean = 



In [ ]:
merge on LAD24CD
retain ITL125CD	ITL125NM	ITL225CD	ITL225NM
dont need ITL325CD	ITL325NM	LAU125CD	LAU125NM   ObjectId

## Renaming columns for ease of use

In [ ]:
# rename columns .str.replace('','')



## Clean dataset

In [ ]:
# Shape of dataset - post-cleaning - to confirm the number of rows and columns
print(f'\nPost-cleaning IMD dataset shape:\n{df_clean.shape}')

# Summary statistics post-cleaning - notice that this only provides stats for numerical data (the IMD rank and IMD decile)
print(f'\nPost-cleaning summary statistics:\n{df_clean.describe()}')

# Save the clean dataset as an Excel file
df_clean.to_excel('', index=False)